# BERT vs. Regex Baseline Comparison

This notebook compares the fine-tuned DistilBERT PII detector against the Week 1 regex baseline, evaluated on the same held-out test set (1,000 examples combining synthetic and real-world data).

In [1]:
import sys
import json
sys.path.insert(0, '../src')

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

from data.generator import Entity
from baseline.regex_detector import detect_pii
from evaluation.metrics import evaluate

## Load test set and fine-tuned model

In [2]:
with open('../data/processed/splits/test.jsonl') as f:
    items = [json.loads(line) for line in f]

print(f"Loaded {len(items)} test examples")

MODEL_PATH = '../models/pii-distilbert'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()
print("Model loaded")

Loaded 1000 test examples


Model loaded


## BERT inference helper

Reconstructs character-span entities from per-token BIO predictions.

In [3]:
def predict_with_bert(text):
    inputs = tokenizer(text, return_tensors="pt", return_offsets_mapping=True, truncation=True, max_length=128)
    offset_mapping = inputs.pop("offset_mapping")[0]
    inputs.pop("token_type_ids", None)

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)[0]

    entities = []
    current_type = None
    current_start = None
    current_end = None

    for pred_id, (start, end) in zip(predictions, offset_mapping):
        start, end = start.item(), end.item()
        if start == end:
            continue
        label = model.config.id2label[pred_id.item()]
        if label == "O":
            if current_type is not None:
                entities.append(Entity(type=current_type, start=current_start, end=current_end, value=text[current_start:current_end]))
                current_type = None
            continue
        prefix, entity_type = label.split("-", 1)
        if prefix == "B" or entity_type != current_type:
            if current_type is not None:
                entities.append(Entity(type=current_type, start=current_start, end=current_end, value=text[current_start:current_end]))
            current_type = entity_type
            current_start = start
            current_end = end
        else:
            current_end = end

    if current_type is not None:
        entities.append(Entity(type=current_type, start=current_start, end=current_end, value=text[current_start:current_end]))

    return entities

## Run both models on the full test set

In [4]:
true_per_example = []
regex_pred_per_example = []
bert_pred_per_example = []

for item in items:
    text = item["text"]
    true_entities = [Entity(**e) for e in item["entities"]]
    true_per_example.append(true_entities)
    regex_pred_per_example.append(detect_pii(text))
    bert_pred_per_example.append(predict_with_bert(text))

print("Done running both models")

Done running both models


## Results comparison

In [5]:
regex_results = evaluate(true_per_example, regex_pred_per_example)
bert_results = evaluate(true_per_example, bert_pred_per_example)

categories = ["EMAIL", "PHONE", "SSN", "CREDIT_CARD", "PERSON_NAME", "ADDRESS", "OVERALL"]

print(f'{"Category":15} {"Regex F1":>10} {"BERT F1":>10} {"Regex R":>10} {"BERT R":>10}')
print("-" * 60)
for cat in categories:
    r = regex_results.get(cat)
    b = bert_results.get(cat)
    r_f1 = r.f1 if r else 0.0
    b_f1 = b.f1 if b else 0.0
    r_r = r.recall if r else 0.0
    b_r = b.recall if b else 0.0
    print(f"{cat:15} {r_f1:>10.3f} {b_f1:>10.3f} {r_r:>10.3f} {b_r:>10.3f}")

Category          Regex F1    BERT F1    Regex R     BERT R
------------------------------------------------------------
EMAIL                1.000      1.000      1.000      1.000
PHONE                0.672      0.975      0.550      0.975
SSN                  0.674      0.976      0.508      0.984
CREDIT_CARD          0.818      0.897      1.000      0.938
PERSON_NAME          0.000      0.972      0.000      0.972
ADDRESS              0.000      0.876      0.000      0.876
OVERALL              0.429      0.948      0.284      0.952


## Takeaway

The fine-tuned DistilBERT model dramatically outperforms the regex baseline overall (F1: 0.948 vs 0.429), with the largest gains exactly where expected: PERSON_NAME (0.0 -> 0.972) and ADDRESS (0.0 -> 0.876), categories regex cannot detect at all since they require contextual language understanding.

Notably, BERT also generalizes better on structured categories that regex nominally supports: PHONE and SSN recall both improve substantially (0.55 -> 0.975 and 0.51 -> 0.98 respectively), because real-world data contains format variations the regex patterns weren't built to handle.

One subtle finding: regex's CREDIT_CARD precision suffers from a structural ambiguity that no regex pattern could fix -- some real-world numeric identifiers (patient IDs, account numbers) are the same length/format as credit card numbers, so length-based pattern matching alone cannot distinguish them. BERT, using contextual understanding, handles this more gracefully (F1: 0.818 -> 0.897).